# Neural Geometry Lab — quickstart

Нотбук для экспериментов в стиле Goodfire: дни недели/месяцы как круги, числа как Fourier/mod-окружности, сложение как геометрический калькулятор.

Запускайте сначала на маленькой модели (`gpt2`, `Qwen/Qwen2.5-0.5B`), потом масштабируйте до Llama/Mistral/Qwen larger.

In [ ]:
# Colab/local install
# !pip install -r requirements.txt
# или:
# !pip install torch transformers accelerate numpy pandas scikit-learn matplotlib scipy tqdm

In [ ]:
from pathlib import Path
import pandas as pd

from nglab import load_lm, extract_activations
from nglab.datasets import make_weekday_dataset, make_number_dataset, make_addition_dataset, make_cyclic_addition_dataset
from nglab.geometry import circular_concept_metrics, fit_fourier_probes
from nglab.plotting import plot_concept_circle, plot_fourier_scores, plot_predicted_mod_circle, plot_layer_heatmap

## 1. Загружаем модель

Для быстрого теста используйте `gpt2`. Для более красивых результатов попробуйте `Qwen/Qwen2.5-0.5B` или более крупные модели.

In [ ]:
MODEL = 'gpt2'
loaded = load_lm(MODEL, device='auto', dtype='auto')

## 2. Дни недели как круг

Берём hidden state самого слова `Monday`/`Tuesday`/... в разных контекстах и смотрим геометрию центроидов.

In [ ]:
df_days_all = make_weekday_dataset()
keep_contexts = list(dict.fromkeys(df_days_all['context'].tolist()))[:4]
df_days = df_days_all[df_days_all['context'].isin(keep_contexts)].reset_index(drop=True)
acts_days = extract_activations(loaded, df_days, layers=[-1], batch_size=8, extraction='target_or_last')
metrics = circular_concept_metrics(acts_days[-1], df_days['value'].to_numpy(), period=7)
metrics

In [ ]:
out = Path('runs/notebook')
out.mkdir(parents=True, exist_ok=True)
img = plot_concept_circle(
    acts_days[-1], df_days['value'].to_numpy(), df_days['label'].tolist(),
    period=7, title=f'Weekday circle — {MODEL}', outpath=out/'weekday_circle.png'
)
from IPython.display import Image
Image(filename=str(img))

## 3. Числа как Fourier/mod-окружности

Для каждого периода `p` обучаем probe из hidden state в `cos(2πn/p), sin(2πn/p)`. Высокий `R²` означает, что в активации линейно читается `n mod p`.

In [ ]:
df_nums_all = make_number_dataset(0, 99)
keep_contexts = list(dict.fromkeys(df_nums_all['context'].tolist()))[:4]
df_nums = df_nums_all[df_nums_all['context'].isin(keep_contexts)].reset_index(drop=True)
acts_nums = extract_activations(loaded, df_nums, layers=[-1], batch_size=8, extraction='target_or_last')
scores, preds = fit_fourier_probes(acts_nums[-1], df_nums['value'].to_numpy(), periods=[2,5,10,20,50,100])
scores

In [ ]:
img = plot_fourier_scores(scores, title=f'Number Fourier probes — {MODEL}', outpath=out/'number_fourier_scores.png')
Image(filename=str(img))

In [ ]:
img = plot_predicted_mod_circle(preds[10], df_nums['value'].to_numpy(), period=10, title='Predicted number positions mod 10', outpath=out/'number_mod10.png')
Image(filename=str(img))

## 4. Сложение: есть ли в hidden state сумма `a+b`?

Это лёгкая версия “geometric calculator”: проверяем, можно ли из активации после `a+b=` декодировать сумму на разных mod-окружностях.

In [ ]:
df_add = make_addition_dataset(max_a=20, max_b=20)
acts_add = extract_activations(loaded, df_add, layers=[-1], batch_size=8, extraction='last')
add_scores, add_preds = fit_fourier_probes(acts_add[-1], df_add['sum'].to_numpy(), periods=[2,5,10,20,50])
add_scores

In [ ]:
img = plot_fourier_scores(add_scores, title=f'Addition Fourier probes — {MODEL}', outpath=out/'addition_fourier_scores.png')
Image(filename=str(img))

## 5. Циклическая арифметика: pre-modulo sum vs output modulo

Goodfire показывает, что для задач вроде “six months after August” модель может сначала считать обычную сумму, а потом переводить её в cyclic domain. Здесь можно посмотреть, что линейно декодируется из hidden state: `premod_sum` или `output_value`.

In [ ]:
df_cyc = make_cyclic_addition_dataset('weekday', max_offset=6)
acts_cyc = extract_activations(loaded, df_cyc, layers=[-1], batch_size=8, extraction='last')
pre_scores, _ = fit_fourier_probes(acts_cyc[-1], df_cyc['premod_sum'].to_numpy(), periods=[2,5,7,10])
out_scores, _ = fit_fourier_probes(acts_cyc[-1], df_cyc['output_value'].to_numpy(), periods=[2,5,7,10])
print('premod_sum')
display(pre_scores)
print('output_value')
display(out_scores)

## 6. Слой за слоем

Для серьёзного исследования запускайте CLI с `--layers all`: он сохранит heatmap и все CSV/NPZ.

```bash
python ng_lab.py numbers --model gpt2 --layers all --start 0 --end 99 --periods 2,5,10,20,50,100 --contexts 3 --outdir runs/gpt2_numbers_sweep
```